# Deep Q-Network (DQN): CartPole with PyTorch
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/10_Reinforcement_Learning/dqn_cartpole_pytorch.ipynb)

Q-learning breaks when states are continuous (a table cannot hold them). DQN replaces the table with a neural network trained on experience replay + a target network - the 2013 algorithm that launched deep RL.

Environment: CartPole-v1 (balance a pole). Runs on free Colab CPU in ~10 minutes.

In [ ]:
!pip install -q gymnasium torch

## 1. Replay buffer + Q-network

In [ ]:
import random
from collections import deque
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn

env = gym.make("CartPole-v1")
device = "cuda" if torch.cuda.is_available() else "cpu"

class ReplayBuffer:
    def __init__(self, cap=50_000):
        self.buf = deque(maxlen=cap)
    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))
    def sample(self, batch):
        batch = random.sample(self.buf, batch)
        s, a, r, s2, d = map(np.array, zip(*batch))
        return (torch.tensor(s, dtype=torch.float32, device=device),
                torch.tensor(a, device=device),
                torch.tensor(r, dtype=torch.float32, device=device),
                torch.tensor(s2, dtype=torch.float32, device=device),
                torch.tensor(d, dtype=torch.float32, device=device))
    def __len__(self):
        return len(self.buf)

q_net = nn.Sequential(nn.Linear(4, 128), nn.ReLU(),
                      nn.Linear(128, 128), nn.ReLU(),
                      nn.Linear(128, 2)).to(device)          # Q(s) -> value per action
target_net = nn.Sequential(nn.Linear(4, 128), nn.ReLU(),
                           nn.Linear(128, 128), nn.ReLU(),
                           nn.Linear(128, 2)).to(device)
target_net.load_state_dict(q_net.state_dict())
optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)

## 2. The training loop

In [ ]:
def act(state, eps):
    if random.random() < eps:
        return env.action_space.sample()
    with torch.no_grad():
        return int(q_net(torch.tensor(state, dtype=torch.float32,
                                         device=device)).argmax())

def optimize(batch=64, gamma=0.99):
    if len(buffer) < batch:
        return None
    s, a, r, s2, d = buffer.sample(batch)
    q_sa = q_net(s).gather(1, a.unsqueeze(1)).squeeze()
    with torch.no_grad():
        target = r + gamma * (1 - d) * target_net(s2).max(1).values
    loss = nn.functional.smooth_l1_loss(q_sa, target)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.item()

buffer = ReplayBuffer()
eps, rewards_history = 1.0, []

for ep in range(250):
    state, _ = env.reset(seed=None)
    total, done = 0.0, False
    while not done:
        action = act(state, eps)
        nxt, reward, term, trunc, _ = env.step(action)
        done = term or trunc
        buffer.push(state, action, reward, nxt, float(done))
        state = nxt; total += reward
        optimize()
    eps = max(0.02, eps * 0.985)                 # explore -> exploit
    if ep % 10 == 0:
        target_net.load_state_dict(q_net.state_dict())
    rewards_history.append(total)
    if ep % 25 == 0:
        print(f"ep {ep:3d}  reward={total:.0f}  eps={eps:.2f}")
    if len(rewards_history) > 30 and np.mean(rewards_history[-30:]) >= 400:
        print("solved-ish!"); break

## 3. Learning curve

In [ ]:
import matplotlib.pyplot as plt
smooth = np.convolve(rewards_history, np.ones(20)/20, mode="valid")
plt.plot(rewards_history, alpha=.25)
plt.plot(smooth, label="20-episode avg")
plt.axhline(475, ls="--", c="r", label="solved (475)")
plt.xlabel("episode"); plt.ylabel("reward"); plt.legend(); plt.show()

## Why each ingredient exists
| Trick | Without it |
|---|---|
| replay buffer | correlated samples destabilize training |
| target network | moving target chases itself |
| epsilon decay | stuck exploiting too early |
| Huber loss | huge TD errors explode gradients |

Next step: PPO notebook shows the policy-gradient alternative that most modern RL actually uses.